# Train MLP Models on Embeddings Dataset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
import os
import torch
import esm
import pandas as pd
from tqdm.notebook import tqdm
import lightning as L
from lightning.pytorch import seed_everything
from dataset import AntibodyDataset
import requests
import numpy as np
from models import MLPModel, MLPModelWithAttention
import json
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

In [ ]:
np.random.seed(7850)

## Load in ESM 8M dataset

In [ ]:
def load_in_esm_dataset(esm_model):
    # Load in ESM 8M dataset
    path = f"../extract_embeddings/data/esm2_{esm_model}_UR50D/dataset.pkl"
    dataset = pd.read_pickle(path)
    return dataset

## Preprocess dataset

In [ ]:
def preprocess(dataset):
    # Remove non-target rows
    dataset = dataset[dataset['Target'] == 'MIT_Target']
    dataset = dataset[dataset['Pred_affinity'].notna()]
    
    # Remove all sequence columns but keep embeddings columns
    sequence_columns = ['Sequence', 'HC', 'LC', 'CDRH1', 'CDRH2', 'CDRH3', 'CDRL1', 'CDRL2', 'CDRL3']
    dataset = dataset.drop(columns=sequence_columns)

    # Remove all columns that don't have embeddings in name and are not Pred_affinity
    dataset = dataset.loc[:, dataset.columns.str.contains('embeddings|Pred_affinity')]
    
    # Only keep 50% of the dataset
    dataset = dataset.sample(frac=0.5)
    
    return dataset

## Split dataset into train, val, and test

In [ ]:
def split_dataset(dataset):
    # Create train, validation, and test datasets using a 70/15/15 split
    train_dataset = dataset.sample(frac=0.7)
    dataset = dataset.drop(train_dataset.index)
    validation_dataset = dataset.sample(frac=0.5)
    test_dataset = dataset.drop(validation_dataset.index)
    print(f"Length of train dataset: {len(train_dataset)}")
    print(f"Length of validation dataset: {len(validation_dataset)}")
    print(f"Length of test dataset: {len(test_dataset)}")
    
    return train_dataset, validation_dataset, test_dataset

In [ ]:
def combine_embeddings(data):
    embeddings_columns = [col for col in data.columns if "embedding" in col]
    all_embeddings = []
    for col in embeddings_columns:
        col_embeddings = []
        for i in range(len(data)):
            col_embeddings.append(data.iloc[i][col])
        col_embeddings = torch.stack(col_embeddings)
        all_embeddings.append(col_embeddings)
    sequence_embeddings = torch.cat(all_embeddings, dim=1)
    pred_affinity = torch.tensor(data['Pred_affinity'].values, dtype=torch.float32)
    return sequence_embeddings.detach().cpu().numpy(), pred_affinity.detach().cpu().numpy()

## Train Random Forest Models

In [ ]:
esm_models = ['8M', '35M', '150M', '650M', '3B']
for esm_model in esm_models:
    print("Running for model: ", esm_model)
    # Create train, test, val datasets
    dataset = load_in_esm_dataset(esm_model)
    dataset = preprocess(dataset)
    train_dataset, validation_dataset, test_dataset = split_dataset(dataset)
    
    X_train, y_train = combine_embeddings(train_dataset)
    X_val, y_val = combine_embeddings(validation_dataset)
    X_test, y_test = combine_embeddings(test_dataset)
    
    print("Training model")
    regr = RandomForestRegressor(n_jobs=-1, random_state=7850, n_estimators=100, max_depth=7, criterion='friedman_mse')
    regr.fit(X_train, y_train)
    
    train_predictions = regr.predict(X_train)
    val_predictions = regr.predict(X_val)
    test_predictions = regr.predict(X_test)
    
    train_corr = spearmanr(train_predictions, y_train)
    train_loss = np.mean((train_predictions - y_train) ** 2)
    val_corr = spearmanr(val_predictions, y_val)
    val_loss = np.mean((val_predictions - y_val) ** 2)
    test_corr = spearmanr(test_predictions, y_test)
    test_loss = np.mean((test_predictions - y_test) ** 2)
    
    print(f"Model: {esm_model}")
    print(f"Train Spearman Correlation: {train_corr.correlation}")
    print(f"Train Loss: {train_loss}")
    print(f"Validation Spearman Correlation: {val_corr.correlation}")
    print(f"Validation Loss: {val_loss}")
    print(f"Test Spearman Correlation: {test_corr.correlation}")
    print(f"Test Loss: {test_loss}")

    os.makedirs("results", exist_ok=True)
    os.makedirs(f"results/random_forest_{esm_model}", exist_ok=True)
    
    with open(f"results/random_forest_{esm_model}.json", "w") as f:
        json.dump({
            "train_corr": train_corr.correlation,
            "train_loss": train_loss,
            "val_corr": val_corr.correlation,
            "val_loss": val_loss,
            "test_corr": test_corr.correlation,
            "test_loss": test_loss
        }, f)